# Interactive PyTorch Training with TorchTPU on GKE Workspaces

This notebook demonstrates how to develop and train machine learning models interactively on Google Cloud TPU v5e using native PyTorch and **TorchTPU (`torch_tpu`)**.

### What you will learn:
1. **Hardware Discovery**: Verifying attached Cloud TPU devices and software runtime.
2. **Tensor Operations**: Creating tensors, casting dtypes (`bfloat16`), and performing computations directly on TPU.
3. **Dataset Generation**: Building a self-contained multi-class classification dataset with PyTorch `DataLoader`.
4. **Model Architecture**: Implementing a standard `nn.Module` MLP classifier.
5. **Interactive Training Loop**: Training the model directly on TPU inside this notebook.
6. **Model Evaluation**: Evaluating accuracy and testing sample predictions.

For multi-core DDP with `torchrun` and multi-host scaling with Kubeflow Trainer, see [torch_tpu_distributed.ipynb](torch_tpu_distributed.ipynb).

## 0. Hardware Discovery, Pre-Flight Lock Verification & TorchTPU Initialization

TorchTPU enables native PyTorch execution on Google Cloud TPUs via the `"tpu"` device string.

`torch_tpu`, PyTorch, and `libtpu` are pre-installed in the custom JupyterLab TPU container image (`jupyterlab:latest-tpu`).

> [!IMPORTANT]
> **TPU Hardware Locks**: Cloud TPU character devices (`/dev/vfio/*`) are claimed exclusively by whichever process first initializes the TPU runtime. The pre-flight check below ensures that no previous session or orphaned worker is holding the hardware before initializing TorchTPU.

In [ ]:
import sys
from pathlib import Path

# Ensure tpu_lock utility is available
for p in [Path.cwd(), Path.cwd() / "examples" / "torch_tpu", Path.cwd().parent]:
    if (p / "tpu_lock.py").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import tpu_lock

# Verify hardware is free and auto-clear any lingering worker processes from previous runs
tpu_lock.preflight_check(is_distributed=False, auto_clear=True)

In [ ]:
import importlib.metadata
import torch
import torch_tpu

print(f"✅ PyTorch version:  {torch.__version__}")
print(f"✅ TorchTPU version: {importlib.metadata.version('torch_tpu')}")
print(f"✅ TorchTPU file:    {torch_tpu.__file__}")

# Initialize TPU device — registers "tpu" device string with PyTorch
device = torch.device("tpu")
print(f"✅ TPU initialization complete. Primary device: {device}")

## 1. Working with Tensors on Cloud TPU

Interacting with a TPU is identical to working with standard PyTorch accelerators (such as CUDA):
1. **Move existing CPU tensors to TPU**: `cpu_tensor.to(device)` or `cpu_tensor.to("tpu")`
2. **Direct creation on TPU**: `torch.randn(..., device="tpu")` (supports `float32` and `bfloat16`)
3. **Computation & Materialization**: Computations happen asynchronously on TPU hardware; retrieve data back to CPU using `.cpu()` or `.item()`

In [ ]:
# 1. Create a tensor on the CPU
cpu_tensor = torch.ones((5, 5))

# 2. Move it to the TPU
tpu_tensor = cpu_tensor.to(device)

# 3. Create a tensor directly on the TPU (bfloat16 for native MXU hardware acceleration)
direct_tpu_tensor = torch.randn((5, 5), device="tpu", dtype=torch.bfloat16)

print(f"cpu_tensor device:        {cpu_tensor.device}, dtype: {cpu_tensor.dtype}")
print(f"tpu_tensor device:        {tpu_tensor.device}, dtype: {tpu_tensor.dtype}")
print(f"direct_tpu_tensor device: {direct_tpu_tensor.device}, dtype: {direct_tpu_tensor.dtype}")

# 4. Perform computations on TPU
result_on_tpu = tpu_tensor + direct_tpu_tensor.to(torch.float32)

# 5. Materialize result back to CPU
result_on_cpu = result_on_tpu.cpu()
print("\n✅ TPU computation successful!")
print(f"Result Checksum (Sum): {result_on_cpu.sum():.4f}")
print(f"Result Tensor Shape:  {result_on_cpu.shape}")

## 2. Synthetic Dataset Generation

We generate a self-contained multi-class classification dataset (8,192 training samples, 1,024 test samples, 784 features, 10 classes, matching MNIST / Fashion-MNIST dimensions).

In [ ]:
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

def generate_synthetic_data(num_samples=8192, num_features=784, num_classes=10, seed=42):
    np.random.seed(seed)
    prototypes = np.random.randn(num_classes, num_features).astype(np.float32)
    labels = np.random.randint(0, num_classes, size=(num_samples,))
    noise = np.random.randn(num_samples, num_features).astype(np.float32) * 0.5
    images = prototypes[labels] + noise
    return torch.tensor(images, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

train_images, train_labels = generate_synthetic_data(num_samples=8192, seed=1)
test_images, test_labels = generate_synthetic_data(num_samples=1024, seed=2)

train_dataset = TensorDataset(train_images, train_labels)
test_dataset = TensorDataset(test_images, test_labels)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Training dataset: {len(train_dataset)} samples, Test dataset: {len(test_dataset)} samples")
print(f"Batch shapes: images {train_images[:256].shape}, labels {train_labels[:256].shape}")

## 3. Model Architecture & Forward Pass

We define a 3-layer MLP (`784 -> 256 -> 128 -> 10`) using standard PyTorch `nn.Module`.
We use `torch.bfloat16` to leverage Cloud TPU v5e Matrix Execution Units (MXUs).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MLPClassifier(nn.Module):
    def __init__(self, in_features=784, hidden1=256, hidden2=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# Instantiate model and transfer to TPU in bfloat16
model = MLPClassifier().to(device=device, dtype=torch.bfloat16)
print(f"Model successfully moved to TPU:")
print(model)

# Test forward pass with sample batch on TPU
sample_batch = train_images[:4].to(device=device, dtype=torch.bfloat16)
sample_logits = model(sample_batch)
print(f"\nSample output logits shape: {sample_logits.shape} (device: {sample_logits.device})")

## 4. Interactive Training Loop on TPU

Train the model directly in this notebook on the workspace's attached Cloud TPU.
The loop uses standard, idiomatic PyTorch:
- `optimizer.zero_grad()`
- `loss.backward()`
- `optimizer.step()`
- `.item()` triggers execution and materializes scalar loss from the TPU.

In [ ]:
import time
import torch.optim as optim

def train_interactive(model, train_loader, epochs=5, lr=0.05):
    optimizer = optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()

    print(f"Starting interactive training on {device} for {epochs} epochs...\n")
    total_start = time.time()

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        running_loss = 0.0
        total_samples = 0

        for images, labels in train_loader:
            images = images.to(device=device, dtype=torch.bfloat16)
            labels = labels.to(device=device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            # .item() materializes the loss scalar from the TPU
            batch_loss = loss.item()
            running_loss += batch_loss * images.size(0)
            total_samples += images.size(0)

        epoch_loss = running_loss / total_samples
        duration = time.time() - epoch_start
        throughput = total_samples / duration
        print(f"Epoch {epoch:2d}/{epochs:2d} | Loss: {epoch_loss:.4f} | Throughput: {throughput:.0f} samples/s ({duration:.2f}s)")

    print(f"\n✅ Interactive training complete in {time.time() - total_start:.2f}s!")

train_interactive(model, train_loader, epochs=5, lr=0.05)

## 5. Model Evaluation & Sample Predictions

Evaluate model accuracy on the held-out test dataset and inspect sample predictions.

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device=device, dtype=torch.bfloat16)
        labels = labels.to(device=device)
        outputs = model(images)
        predictions = outputs.argmax(dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = 100.0 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%\n")

# Display sample predictions vs ground truth
sample_images = test_images[:5].to(device=device, dtype=torch.bfloat16)
sample_labels = test_labels[:5]
with torch.no_grad():
    preds = model(sample_images).argmax(dim=-1).cpu()

print("Sample Predictions:")
for i in range(5):
    match = "OK" if preds[i] == sample_labels[i] else "MISMATCH"
    print(f"  Sample {i+1}: Predicted = {preds[i].item()}, True = {sample_labels[i].item()} [{match}]")

## 6. Next Steps: Multi-Core Distributed Training

To scale this model to all attached TPU cores with PyTorch DistributedDataParallel (`torchrun`), open [`torch_tpu_distributed.ipynb`](torch_tpu_distributed.ipynb).

> [!IMPORTANT]
> **Releasing the TPU Lock**: Because this notebook initialized TPU hardware in its active Python kernel, the kernel process maintains exclusive access to `/dev/vfio/*`. Before running `torchrun` in `torch_tpu_distributed.ipynb`, **release the hardware lock** using Section 7 below or by restarting this kernel (`Kernel` -> `Restart Kernel`).

## 7. Releasing TPU Hardware Locks (Cleanup)

Cloud TPU character devices (`/dev/vfio/*`) are held exclusively by the active Python kernel process via libtpu/PJRT. Standard Python garbage collection (`del model`, `gc.collect()`) **cannot** release OS-level hardware file descriptors.

To release the TPU hardware for another notebook (such as [`torch_tpu_distributed.ipynb`](torch_tpu_distributed.ipynb)) or another user session:
- **UI Option**: In JupyterLab, click **Kernel** -> **Restart Kernel...** (or **Shut Down Kernel**). In VS Code, click **Restart** at the top.
- **Programmatic Option**: Run the cell below to restart the kernel cleanly. All cell outputs and plots remain visible on screen, but the kernel process exits and restarts without holding the TPU lock.

In [ ]:
# Execute this cell when you are finished to release the TPU hardware lock
# for other notebooks (e.g. torch_tpu_distributed.ipynb) or another user session.
import tpu_lock

print("🔄 Releasing TPU hardware lock by restarting the kernel...")
tpu_lock.release_current_kernel(restart=True)